# LastFM (HetRec 2011) — Behavior Feature Computation

This notebook computes the **6 behavior features** for the LastFM dataset:

1. Activity
2. Popularity
3. Influence
4. Community Participation
5. Reciprocity
6. Reputation / Trustworthiness


In [2]:
# If needed, install dependencies first (uncomment):
# !pip install pandas networkx python-louvain --quiet

import pandas as pd
import networkx as nx
import community as community_louvain  # from python-louvain package

DATA_DIR = ""  # change this to the folder containing the .dat 
# the folder in which this ipynb file is present, it already contains the .dat files which is this http://localhost:8888/notebooks/Downloads/BehaviourFeatures/LastFM_Behavior_Features.ipynb


## Step 1 — Load the raw files

In [3]:
user_friends = pd.read_csv(DATA_DIR + "user_friends.dat", sep="\t")
user_artists = pd.read_csv(DATA_DIR + "user_artists.dat", sep="\t")
user_tagged  = pd.read_csv(DATA_DIR + "user_taggedartists.dat", sep="\t")

print("user_friends:", user_friends.shape)
print("user_artists:", user_artists.shape)
print("user_tagged:", user_tagged.shape)

user_friends.head()


user_friends: (25434, 2)
user_artists: (92834, 3)
user_tagged: (186479, 6)


,userID,friendID
0,2,275
1,2,428
2,2,515
3,2,761
4,2,831


## Feature 1 — Activity

How engaged a user is on the platform: number of distinct artists listened to, plus number of tags they've applied.
Higher activity = more behavioral data available for that user, which matters for recommendation quality (cold-start risk if low).


In [4]:
artists_listened = user_artists.groupby("userID")["artistID"].nunique().rename("artists_listened")
tags_applied     = user_tagged.groupby("userID")["tagID"].count().rename("tags_applied")

activity = pd.concat([artists_listened, tags_applied], axis=1).fillna(0)
activity["activity_score"] = activity["artists_listened"] + activity["tags_applied"]

activity.sort_values("activity_score", ascending=False).head(10)


,artists_listened,tags_applied,activity_score
userID,,,
1672,50,2609,2659
1380,50,2075,2125
1249,50,2024,2074
1929,50,1876,1926
1832,50,1863,1913
236,50,1687,1737
387,50,1677,1727
1623,50,1674,1724
1679,50,1637,1687


## Step 2 — Build the friend network graph

Needed for Popularity, Influence, Community Participation, and Reciprocity.


In [5]:
G = nx.from_pandas_edgelist(user_friends, source="userID", target="friendID", create_using=nx.Graph())
print("Nodes:", G.number_of_nodes(), "| Edges:", G.number_of_edges())


Nodes: 1892 | Edges: 12717


## Feature 2 — Popularity

Number of friends a user has (degree in the friend network). This is your "followers/listeners" equivalent for LastFM.


In [6]:
popularity = pd.Series(dict(G.degree()), name="popularity_degree")
popularity.sort_values(ascending=False).head(10)


1543    119
1281    110
831     106
179      97
1503     95
1023     91
1895     90
405      90
1300     89
390      88
Name: popularity_degree, dtype: int64

## Feature 3 — Influence

Two standard measures:
- **PageRank**: how influential a user is based on who is connected to them (not just how many friends, but how well-connected those friends are)
- **Betweenness Centrality**: how often a user sits "between" other users on the shortest path — i.e. how much of a bridge/connector they are


In [7]:
pagerank    = pd.Series(nx.pagerank(G), name="pagerank")
betweenness = pd.Series(nx.betweenness_centrality(G), name="betweenness")

influence = pd.concat([pagerank, betweenness], axis=1)
influence.sort_values("pagerank", ascending=False).head(10)


,pagerank,betweenness
1543,0.005246,0.066527
78,0.005238,0.029337
1281,0.004735,0.054549
1258,0.004230,0.043025
1210,0.003870,0.019085
831,0.003783,0.048691
298,0.003060,0.023113
1488,0.002943,0.016479
1213,0.002849,0.024167
1597,0.002785,0.021753


## Feature 4 — Community Participation

Uses the **Louvain algorithm** (same modularity-optimization method Gephi uses under the hood) to assign each user to a community/cluster.
This gives you a `community_id` per user — cross-check this against Gephi's "Modularity Class" column to confirm they match.


In [8]:
partition = community_louvain.best_partition(G)
community_col = pd.Series(partition, name="community_id")

print("Number of communities found:", community_col.nunique())
community_col.head(10)


Number of communities found: 32


2       9
275     9
428     9
515     9
761     9
831     9
909     9
1209    9
1210    9
1230    9
Name: community_id, dtype: int64

## Feature 5 — Reciprocity

Fraction of a user's outgoing friend links that are mutual (i.e. A→B exists AND B→A exists).
Important to compute this from the **raw directed edge list**, before it gets treated as an undirected graph — otherwise you lose the asymmetry information entirely.


In [9]:
edge_set = set(zip(user_friends["userID"], user_friends["friendID"]))

def fraction_mutual(u, out_edges_by_user):
    out_edges = out_edges_by_user.get(u, [])
    if not out_edges:
        return 0.0
    mutual = sum(1 for (a, b) in out_edges if (b, a) in edge_set)
    return mutual / len(out_edges)

# group outgoing edges per user once (faster than filtering edge_set repeatedly)
out_edges_by_user = {}
for u, f in zip(user_friends["userID"], user_friends["friendID"]):
    out_edges_by_user.setdefault(u, []).append((u, f))

reciprocity = pd.Series(
    {u: fraction_mutual(u, out_edges_by_user) for u in user_friends["userID"].unique()},
    name="reciprocity"
)

reciprocity.sort_values(ascending=False).head(10)


2     1.0
3     1.0
4     1.0
5     1.0
6     1.0
7     1.0
8     1.0
9     1.0
10    1.0
11    1.0
Name: reciprocity, dtype: float64

## Feature 6 — Reputation / Trustworthiness

Proxy: total plays a user has generated across all artists (`weight` column in `user_artists.dat`).
This reflects "popularity among listeners" — a heavier, more committed listener carries more behavioral trust signal.


In [10]:
reputation = user_artists.groupby("userID")["weight"].sum().rename("total_plays")
reputation.sort_values(ascending=False).head(10)


userID
757     480039
2000    468409
1418    416349
1642    388251
1094    379125
1942    348527
2071    338400
2031    329980
514     329782
387     322661
Name: total_plays, dtype: int64

## Step 3 — Merge everything into one feature table

One row per user, all 6 behavior features (as multiple sub-columns where relevant).


In [11]:
final_features = pd.concat(
    [activity, popularity, pagerank, betweenness, community_col, reciprocity, reputation],
    axis=1
)
final_features.index.name = "userID"

print("Final shape:", final_features.shape)
final_features.head(10)


Final shape: (1892, 9)


,artists_listened,tags_applied,activity_score,popularity_degree,pagerank,betweenness,community_id,reciprocity,total_plays
userID,,,,,,,,,
2,50,45,95,13,0.000590,0.001337,9,1.0,168737
3,50,56,106,7,0.000456,0.000100,9,1.0,20501
4,50,61,111,10,0.000414,0.000072,9,1.0,26930
5,50,43,93,7,0.000407,0.000807,9,1.0,13159
6,50,14,64,5,0.000285,0.000282,9,1.0,1011
7,50,3,53,18,0.000497,0.000135,2,1.0,135399
8,50,20,70,11,0.000334,0.000170,4,1.0,36279
9,50,129,179,3,0.000184,0.000011,5,1.0,25597
10,50,24,74,5,0.000268,0.000085,6,1.0,28523


## Step 4 — Save to CSV

Export this so it's ready to merge with the other datasets' behavior feature tables later.


In [12]:
final_features.to_csv("lastfm_behavior_features.csv")
print("Saved: lastfm_behavior_features.csv")


Saved: lastfm_behavior_features.csv
